# UE24CS352A - Machine Learning Mini-Project
## Sentiment Analysis of Microblog Posts on Attitude Towards Vaccination Since the Distribution of COVID-19 Vaccines

**Team Members:**
- **Navaneeth Tanuboddi** (SRN: `PES1UG24CS709`)
- **Krrish P Raju** (SRN: `PES1UG24CS708`)

---
### Project Abstract & Objectives
In this project, we reproduce and implement the machine learning pipeline for microblog sentiment analysis on COVID-19 vaccination:
1. **Preprocessing & N-gram Tokenization** (Unigram, Bigram, Trigram, TF-IDF)
2. **Classical Classifiers**: Naive Bayes (Laplace smoothing & linear interpolation), Support Vector Machines (SVM), and Regularized Logistic Regression
3. **Generative Modeling**: Bigram Language Model Classifier
4. **Deep Sequence Modeling**: Recurrent Neural Network / LSTM with embedding layers
5. **Temporal Trend Analysis**: Tracking public sentiment evolution against national vaccine administration trajectories.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add source path
sys.path.append(os.path.abspath('..'))
from src.preprocessing import clean_text, tokenize
from src.util import compute_metrics, format_metrics_table, load_dataset_csv
from src.naive_bayes import MultinomialNaiveBayes, InterpolatedNaiveBayes
from src.svm_classifier import SVMClassifier
from src.logistic_regression import LogisticRegressionClassifier
from src.bigram_model import BigramLanguageModelClassifier
from src.rnn_lstm_model import LSTMSentimentClassifier

print("All modules loaded successfully!")

### 1. Load Dataset Splits
We load the curated dataset splits (Train: 2100, Dev: 850, Test: 850) matching the reference study.

In [ ]:
data_dir = '../data/processed'
train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
dev_df = pd.read_csv(os.path.join(data_dir, 'dev.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))

print(f"Train size: {len(train_df)}, Dev size: {len(dev_df)}, Test size: {len(test_df)}")
train_df.head(5)

### 2. Preprocessing & N-Gram Tokenization Demo
Demonstrating clean_text, unigram, bigram, and trigram feature extraction.

In [ ]:
sample_post = "今天去社区医院接种了新冠疫苗，整体感觉很棒，除了针口稍微有点胀痛，完全没有其他异常反应，点赞！"
cleaned = clean_text(sample_post)
print("Cleaned Text:", cleaned)
print("Unigram tokens:", tokenize(cleaned, method='unigram')[:8])
print("Bigram tokens:", tokenize(cleaned, method='bigram')[:8])
print("Trigram tokens:", tokenize(cleaned, method='trigram')[:8])

### 3. Model Training & Validation (Table 1 Replication)
Training Naive Bayes, SVM, Logistic Regression, Bigram Model, and LSTM.

In [ ]:
X_train, y_train = train_df['text'].tolist(), train_df['label'].tolist()
X_dev, y_dev = dev_df['text'].tolist(), dev_df['label'].tolist()

# Initialize models
models = {
    'Naive Bayes (Bigram)': MultinomialNaiveBayes(tokenization='bigram'),
    'Naive Bayes (Interpolated)': InterpolatedNaiveBayes(lmbda=0.8),
    'SVM (TF-IDF)': SVMClassifier(tokenization='bigram', C=2.0),
    'Logistic Regression': LogisticRegressionClassifier(tokenization='bigram', C=1.0),
    'Bigram Model': BigramLanguageModelClassifier(alpha=1.0),
    'Neural LSTM': LSTMSentimentClassifier(vocab_size=3000, embed_dim=64, hidden_dim=64)
}

results = []
for name, clf in models.items():
    print(f"Training {name}...")
    clf.fit(X_train, y_train)
    preds = clf.predict(X_dev)
    m = compute_metrics(y_dev, preds)
    m['Model'] = name
    results.append(m)

res_df = pd.DataFrame(results)[['Model', 'accuracy', 'precision', 'recall', 'f1']]
res_df.columns = ['Model', 'Dev Accuracy', 'Precision', 'Recall', 'F1 Score']
res_df

### 4. Temporal Trend Analysis & Figure 1 Plot
Evaluating how public sentiment evolved across December 2020 through April 2021 as national vaccine doses were rolled out.

In [ ]:
from src.trend_analysis import plot_sentiment_vs_vaccine_doses

timeline_csv = '../data/processed/vaccine_doses_timeline.csv'
fig_out = '../report/figures/figure1_sentiment_trend.png'
plot_sentiment_vs_vaccine_doses(timeline_csv, fig_out)

img = plt.imread(fig_out)
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.show()

### 5. Interactive Live Inference Test
Enter any microblog sentence below to run real-time sentiment predictions across models.

In [ ]:
def test_custom_text(text):
    print(f"Post: {text}\n" + "-" * 50)
    for name, clf in models.items():
        if hasattr(clf, 'predict_proba'):
            p = clf.predict_proba(text)
            p1 = p[1] if isinstance(p, dict) else (p[0][1] if hasattr(p, 'shape') else 0.5)
            print(f"{name:<26}: {'POSITIVE' if p1 >= 0.5 else 'NEGATIVE'} (confidence: {max(p1, 1-p1)*100:.1f}%)")
        else:
            pred = clf.predict([text])[0]
            print(f"{name:<26}: {'POSITIVE' if pred==1 else 'NEGATIVE'}")

test_custom_text("打了第一针新冠疫苗，没有任何不良反应，相信国家！")
print()
test_custom_text("打完疫苗后手臂酸痛了好几天，甚至抬不起来，难受。")